## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

In [2]:
import sys
import subprocess

# This forces Python to install pip into the current environment
subprocess.check_call([sys.executable, "-m", "ensurepip", "--default-pip"])

# Now upgrade it to be safe
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

0

In [9]:
%pip install ddgs

Note: you may need to restart the kernel to use updated packages.


c:\Users\prasa\OneDrive\Documents\Legal_Professional\Agent_MCP_Udemy\agents\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\prasa\OneDrive\Documents\Legal_Professional\Agent_MCP_Udemy\agents\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\prasa\OneDrive\Documents\Legal_Professional\Agent_MCP_Udemy\agents\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [49]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
import ddgs
from agents import function_tool
import re
import json

In [12]:
load_dotenv(override=True)

False

In [13]:
# 1. Point the SDK to your local Ollama server
# Ollama's OpenAI-compatible endpoint is /v1
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"  # A placeholder is required

In [16]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."


@function_tool
def local_web_search(query: str) -> str:
    """
    Searches the web for the given query and returns a concatenated 
    string of the top results.
    """
    print(f"--- Locally Searching for: {query} ---")
    results = ddgs().text(query, max_results=5)
    return "\n\n".join([f"Source: {r['href']}\nContent: {r['body']}" for r in results])

# Now update your search_agent to use THIS tool
search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[local_web_search], # Swapped WebSearchTool for local_web_search
    model="ministral-3:8b-cloud",        # Use the name that worked in your test
    model_settings=ModelSettings(tool_choice="auto"),
)

In [17]:
message = "Latest AI Agent frameworks in 2026"

with trace("Ollama_Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

--- Locally Searching for: latest AI agent frameworks 2026 ---
--- Locally Searching for: latest AI agent frameworks 2026 ---
--- Locally Searching for: top AI agent frameworks predicted for 2026 ---
--- Locally Searching for: trends in AI agent frameworks 2026 ---


Apologies, unable to retrieve current web search results. Here’s a **general update** based on trends as of mid-2024, extrapolated for 2026:

**Key AI Agent Frameworks Emerging by 2026:**
Autonomous AI agents are evolving toward **multi-modal, modular architectures** with stronger **tool integration** and **human-like reasoning**. Frameworks like **LangGraph**, **AutoGen**, and **Creative Machine** (by Google) are expected to dominate, emphasizing **agent orchestration** and **dynamic workflows**. **Foundation models** (e.g., GPT-4o, Llama 3) will underpin these, with specialized agents for **automation, creative tasks, and decision-making** via plugins/APIs.

**Upcoming Innovations:**
**Self-improving agents** (e.g., **ReAct**, **Plan-and-Execute**) will gain traction, combining **reinforcement learning** with **large language models**. **Federated learning** and **edge deployment** will enable scalable, privacy-preserving agents. Startups like **Runway ML** and **DeepScribe** are pushing **video/audio agent frameworks**, while **AI middleware** (e.g., **Agentic Cloud**) will unify disparate tools. **Ethical guardrails** and **explainability** will become standard features.

### We will now use Structured Outputs, and include a description of the fields

In [30]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

# 1. Update Instructions to be more strict
# Match these instructions EXACTLY to your Pydantic class field names
PLANNER_INSTRUCTIONS = f"""You are a helpful research assistant. 
Given a query, come up with {HOW_MANY_SEARCHES} web search terms.

You MUST respond with a JSON object following this EXACT structure:
{{
  "searches": [
    {{
      "reason": "why this search is important",
      "query": "the search term"
    }}
  ]
}}

CRITICAL: Use the key "searches", NOT "search_terms". 
Return ONLY the raw JSON. No markdown, no backticks, no preamble."""

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    # This allows the model to be a little "loose" with the naming
    searches: list[WebSearchItem] = Field(
        alias="search_terms", 
        validation_alias="search_terms", 
        description="A list of web searches..."
    )
    
    class Config:
        populate_by_name = True

planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_INSTRUCTIONS, # Use the strict version
    model="ministral-3:8b-cloud", 
    # output_type=WebSearchPlan,
    # 2. Add 'json_object' format if your SDK version supports it
    model_settings=ModelSettings(response_format={"type": "json_object"})
)

c:\Users\prasa\OneDrive\Documents\Legal_Professional\Agent_MCP_Udemy\agents\.venv\Lib\site-packages\pydantic\_internal\_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


In [31]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

```json
{
  "searches": [
    {
      "reason": "Identifies the latest AI agent frameworks released or announced in 2025, including research papers, conference talks, or official documentation.",
      "query": "latest AI agent frameworks 2025 research papers or conference announcements"
    },
    {
      "reason": "Highlights industry leaders and their updates on AI agent frameworks for 2025, including roadmaps, new features, or partnerships.",
      "query": "AI agent frameworks 2025 updates from Microsoft, Google, Anthropic, or Mistral AI"
    },
    {
      "reason": "Focuses on emerging open-source or commercial tools and libraries designed for AI agents, including benchmarks or comparisons.",
      "query": "top AI agent frameworks 2025 GitHub trends or open-source project comparisons"
    }
  ]
}
```


In [58]:
# @function_tool
# def send_email(subject: str, html_body: str) -> Dict[str, str]:
#     """ Send out an email with the given subject and HTML body """
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("pgaitond@gmail.com") # Change this to your verified email
#     to_email = To("pgaitond@gmail.com") # Change this to your email
#     content = Content("text/html", html_body)
#     mail = Mail(from_email, to_email, subject, content).get()
#     sg.client.mail.send.post(request_body=mail)
#     return "success"


async def send_email(report_text: str):
    """
    Fixed: accepting 'report_text' as a string instead of an object.
    """
    print("Writing email...")
    # Assuming email_agent is defined elsewhere in your notebook
    # We pass the string directly now instead of report.markdown_report
    # result = await Runner.run(email_agent, report_text)
    print("Email sent")
    return report_text    

In [59]:
send_email

<function __main__.send_email(report_text: str)>

In [34]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""



email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="ministral-3:8b-cloud", # Updated model
)



In [56]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="ministral-3:8b-cloud", # Updated model
    # output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [54]:
async def plan_searches(query: str) -> WebSearchPlan:
    print(f"Planning searches for: {query}")
    
    # Run the agent
    result = await Runner.run(planner_agent, query)
    
    # 4. Extraction Logic
    # First, try the standard SDK way
    all_items = result.new_items + result.to_input_list()
    for item in all_items:
        if hasattr(item, 'tool_call') and item.tool_call.function.name == "submit_search_plan":
            args = item.tool_call.function.arguments
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except json.JSONDecodeError:
                    continue
            if isinstance(args, dict) and 'searches' in args:
                return WebSearchPlan(searches=args['searches'])
            
    # 5. Robust Fallback (Fixes the current error)
    # If the model put the JSON in the text/final_output instead of a proper tool call
    content = result.final_output or ""
    
    # Extract JSON from potential Markdown blocks or raw text
    # This regex looks for everything between the first { and the last }
    json_match = re.search(r"(\{.*\})", content, re.DOTALL)
    if json_match:
        try:
            clean_json = json_match.group(1)
            data = json.loads(clean_json)
            if "searches" in data:
                print("Extracted plan from text fallback.")
                return WebSearchPlan(searches=data['searches'])
        except Exception as e:
            print(f"Fallback parsing failed: {e}")

    raise ValueError(f"Model failed to call the 'submit_search_plan' tool. Output was: {content}")

# --- Usage ---
# query = "Latest AI Agent frameworks 2025"
# plan = await plan_searches(query)
# print(plan.searches[0].query)

In [51]:
# async def plan_searches(query: str):
#     """ Use the planner_agent to plan which searches to run for the query """
#     print("Planning searches...")
#     result = await Runner.run(planner_agent, f"Query: {query}")
#     print(f"Will perform {len(result.final_output.searches)} searches")
#     return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [52]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [60]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




Starting research...
Planning searches for: Latest AI Agent frameworks in 2025
Extracted plan from text fallback.
Searching...
--- Locally Searching for: latest AI agent frameworks 2025 comparisons and reviews ---
--- Locally Searching for: AI agent framework benchmarks and use cases 2025 industry reports ---
--- Locally Searching for: upcoming AI agent frameworks 2025 research papers ---
--- Locally Searching for: AI agent frameworks developer previews 2025 ---
--- Locally Searching for: latest AI agent frameworks 2025 comparisons ---
--- Locally Searching for: latest AI agent frameworks 2025 reviews ---
--- Locally Searching for: upcoming AI agent frameworks 2025 research papers ---
--- Locally Searching for: AI agent framework benchmarks use cases 2025 ---
--- Locally Searching for: top AI agent frameworks 2025 ---
--- Locally Searching for: AI agent frameworks benchmarks performance metrics real-world applications 2025 ---
--- Locally Searching for: AI agent frameworks industry rep

### As always, take a look at the trace

https://platform.openai.com/traces